In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# RGB-DCT Terminal-Gradient-V1 fixed six-slot run

Run all once. The pinned source creates two fixed sources with OFF, original SINGLE49, and TERMINAL46, for exactly six retained MP4 score slots. Setup failures retain a separate six-slot setup_slots.json; once setup succeeds, the runner writes result.json before it starts either fresh case worker. The user performs the Colab/GPU/Drive run.

A complete fixed run can establish only the behavior of this two-source, fixed-key mechanism trial. It cannot establish low FPR, population detection, payload performance, a video-quality pass, or general effectiveness. Failed, missing, and engineering-invalid slots remain in the fixed denominator.


In [ ]:
from pathlib import Path
import datetime, json, sys

SOURCE_SHA = '9f132aa085bc58cb7133e76dfd79b59bb2ce3dd9'
CASE_IDS = ('eval_clock_s2431', 'eval_umbrella_s2432')
ARMS = ('OFF', 'SINGLE49', 'TERMINAL46')
FIXED_DENOMINATOR = {'sources': 2, 'arms_per_source': 3, 'mp4_score_slots': 6, 'frames': 1086}
OUTPUT_PARENT = Path('/content/drive/MyDrive/Video-WM/RGB-DCT-Terminal-Gradient-V1')
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT = OUTPUT_PARENT / stamp
OUTPUT.mkdir(exist_ok=False)
SETUP_SLOTS_PATH = OUTPUT / 'setup_slots.json'
SETUP_RECEIPT_PATH = OUTPUT / 'setup_receipt.json'
setup_slots = {
    'status': 'GLOBAL_SETUP_NOT_COMPLETED',
    'source_sha': SOURCE_SHA,
    'fixed_denominator': FIXED_DENOMINATOR,
    'cases': {
        case_id: {
            'status': 'NOT_RUN_GLOBAL_SETUP_FAILURE',
            'slots': {
                arm: {
                    'status': 'NOT_RUN_GLOBAL_SETUP_FAILURE',
                    'reason': 'SETUP_NOT_COMPLETED',
                    'path': None,
                    'score': None,
                    'decision': None,
                }
                for arm in ARMS
            },
        }
        for case_id in CASE_IDS
    },
}
SETUP_SLOTS_PATH.write_text(json.dumps(setup_slots, indent=2) + '\n', encoding='utf-8')
SETUP_RECEIPT_PATH.write_text(json.dumps({
    'status': 'SETUP_STARTED',
    'source_sha': SOURCE_SHA,
    'output_dir': str(OUTPUT),
    'python': sys.version,
    'executable': sys.executable,
}, indent=2) + '\n', encoding='utf-8')
print('fresh output:', OUTPUT, flush=True)


In [ ]:
import importlib.metadata
import os
import subprocess

SETUP_LOG = OUTPUT / 'setup.log'

def write_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)


def mark_setup_failure(reason, *, command=None, returncode=None):
    retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
    retained['status'] = 'GLOBAL_SETUP_FAILED'
    retained['reason'] = reason
    for case in retained['cases'].values():
        case['status'] = 'NOT_RUN_GLOBAL_SETUP_FAILURE'
        for slot in case['slots'].values():
            slot['status'] = 'NOT_RUN_GLOBAL_SETUP_FAILURE'
            slot['reason'] = reason
    write_json(SETUP_SLOTS_PATH, retained)
    failure = {'status': 'SETUP_FAILED', 'reason': reason, 'command': command, 'returncode': returncode}
    write_json(OUTPUT / 'setup_failure.json', failure)
    receipt = json.loads(SETUP_RECEIPT_PATH.read_text(encoding='utf-8'))
    receipt.update(failure)
    write_json(SETUP_RECEIPT_PATH, receipt)


def logged_run(command, *, cwd=None, env=None, check=True, log_path=SETUP_LOG):
    with log_path.open('a', encoding='utf-8') as log:
        line = 'COMMAND ' + repr(command) + '\n'
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
        child = subprocess.Popen(
            command, cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        for line in child.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = child.wait()
        log.write('EXIT ' + str(returncode) + '\n')
        log.flush()
    if check and returncode:
        reason = 'SETUP_COMMAND_EXIT_' + str(returncode)
        mark_setup_failure(reason, command=command, returncode=returncode)
        raise subprocess.CalledProcessError(returncode, command)
    return subprocess.CompletedProcess(command, returncode)


def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

print('Python:', sys.version, flush=True)
print('Executable:', sys.executable, flush=True)
logged_run([sys.executable, '-m', 'pip', '--version'])
logged_run(['apt-get', 'update', '-qq'])
logged_run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])
if version('torch') is None:
    logged_run([
        sys.executable, '-m', 'pip', 'install',
        'torch==2.11.0', 'torchvision',
        '--index-url', 'https://download.pytorch.org/whl/cu128',
    ])
logged_run([
    sys.executable, '-m', 'pip', 'install',
    'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy',
    'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow',
])


In [ ]:
environment_check_code = r'''import importlib.metadata
import inspect
import json
import os
import shutil
import sys
import traceback
from pathlib import Path

receipt_path = Path(os.environ['RGB_DCT_SETUP_ENVIRONMENT_RECEIPT'])
packages = (
    'torch', 'torchvision', 'diffusers', 'transformers', 'accelerate',
    'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow',
)
receipt = {
    'status': 'FAILED',
    'python': sys.version,
    'executable': sys.executable,
    'packages': {},
    'ffmpeg': shutil.which('ffmpeg'),
    'ffprobe': shutil.which('ffprobe'),
    'torch_version_policy': 'record_and_api_check',
    'cuda_build_suffix_policy': 'record_only',
}
for name in packages:
    try:
        receipt['packages'][name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        receipt['packages'][name] = None
try:
    import torch
    import diffusers
    from diffusers import AutoencoderKLWan, UniPCMultistepScheduler, WanPipeline
    from torch.utils.checkpoint import checkpoint

    torch_version = str(torch.__version__)
    receipt.update({
        'torch_version': torch_version,
        'torch_public_version': torch_version.split('+', 1)[0],
        'torch_build_suffix': torch_version.split('+', 1)[1] if '+' in torch_version else None,
        'torch_cuda_runtime': torch.version.cuda,
        'diffusers_version': diffusers.__version__,
        'cuda_available': torch.cuda.is_available(),
        'cuda_device_count': torch.cuda.device_count(),
        'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        'cuda_device_capability': list(torch.cuda.get_device_capability(0)) if torch.cuda.is_available() else None,
        'cuda_bfloat16_supported': bool(torch.cuda.is_bf16_supported()) if torch.cuda.is_available() else False,
        'non_reentrant_checkpoint_parameter': 'use_reentrant' in inspect.signature(checkpoint).parameters,
        'saved_tensors_hooks_callable': callable(getattr(torch.autograd.graph, 'saved_tensors_hooks', None)),
        'wan_pipeline_import': WanPipeline is not None,
        'wan_vae_import': AutoencoderKLWan is not None,
    })
    assert receipt['ffmpeg'] and receipt['ffprobe'], 'ffmpeg and ffprobe are required'
    assert diffusers.__version__ == '0.40.0', diffusers.__version__
    assert receipt['cuda_available'], 'CUDA torch required'
    assert receipt['non_reentrant_checkpoint_parameter'], 'non-reentrant checkpoint API unavailable'
    assert receipt['saved_tensors_hooks_callable'], 'saved_tensors_hooks API unavailable'

    x = torch.ones(4, device='cuda', dtype=torch.float32, requires_grad=True)
    y = checkpoint(lambda value: (value * value).sum(), x, use_reentrant=False)
    y.backward()
    assert x.grad is not None and bool(torch.isfinite(x.grad).all()), 'CUDA autograd/checkpoint smoke failed'

    scheduler = UniPCMultistepScheduler(num_train_timesteps=1000)
    scheduler.set_timesteps(4, device='cuda')
    sample = torch.zeros((1, 1, 2, 2), device='cuda', dtype=torch.float32)
    model_output = torch.ones_like(sample, requires_grad=True)
    previous = scheduler.step(model_output, scheduler.timesteps[0], sample, return_dict=False)[0]
    scheduler_gradient = torch.autograd.grad(previous.sum(), model_output)[0]
    assert bool(torch.isfinite(scheduler_gradient).all()), 'differentiable UniPC step smoke failed'
    receipt['cuda_autograd_checkpoint_smoke'] = True
    receipt['differentiable_unipc_step_smoke'] = True
    receipt['status'] = 'VALID'
except Exception as exc:
    receipt['reason'] = f'{type(exc).__name__}: {exc}'
    receipt['traceback'] = traceback.format_exc()
finally:
    receipt_path.write_text(json.dumps(receipt, indent=2) + '\n', encoding='utf-8')
if receipt['status'] != 'VALID':
    raise RuntimeError(receipt['reason'])
'''
setup_environment_path = OUTPUT / 'setup_environment_receipt.json'
check_env = os.environ.copy()
check_env['RGB_DCT_SETUP_ENVIRONMENT_RECEIPT'] = str(setup_environment_path)
completed = logged_run(
    [sys.executable, '-u', '-c', environment_check_code],
    env=check_env,
    check=False,
)
if completed.returncode:
    if setup_environment_path.exists():
        setup_environment = json.loads(setup_environment_path.read_text(encoding='utf-8'))
        reason = 'ENVIRONMENT_CHECK:' + setup_environment.get('reason', 'fresh child failed')
    else:
        reason = 'ENVIRONMENT_CHECK:fresh child produced no receipt'
    mark_setup_failure(reason, command=[sys.executable, '-u', '-c', '<environment_check_code>'], returncode=completed.returncode)
    raise subprocess.CalledProcessError(completed.returncode, [sys.executable, '-u', '-c', '<environment_check_code>'])
print('fresh-child environment receipt:', setup_environment_path, flush=True)


In [ ]:
REPO = Path('/content/SC-SSTW-RGB-DCT-TERMINAL-GRADIENT-' + stamp)
try:
    logged_run(['git', 'clone', '--filter=blob:none', 'https://github.com/RICHAAARC/SC-SSTW.git', str(REPO)])
    logged_run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', 'dev/rgb-dct-terminal-gradient-v1'])
    logged_run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_SHA])
    actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    attached = subprocess.run(
        ['git', '-C', str(REPO), 'symbolic-ref', '-q', 'HEAD'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    ).returncode == 0
    porcelain = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
    if actual != SOURCE_SHA:
        raise RuntimeError('immutable source SHA readback mismatch')
    if attached:
        raise RuntimeError('source checkout is not detached')
    if porcelain:
        raise RuntimeError('source checkout is not clean')
    source_files = {
        'protocol': REPO / 'docs/rgb_dct_terminal_gradient_v1_protocol.md',
        'config': REPO / 'experiments/wan_state_clock/configs/rgb_dct_terminal_gradient_v1.json',
        'runner': REPO / 'experiments/wan_state_clock/rgb_dct_terminal_gradient_run.py',
    }
    import hashlib
    source_receipt = {
        'status': 'VALID',
        'expected_sha': SOURCE_SHA,
        'actual_sha': actual,
        'detached_head': not attached,
        'clean_checkout': not bool(porcelain),
        'repo': str(REPO),
        'files': {
            name: {'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
            for name, path in source_files.items()
        },
    }
    write_json(OUTPUT / 'source_receipt.json', source_receipt)
except Exception as exc:
    reason = f'SOURCE_CHECKOUT:{type(exc).__name__}: {exc}'
    mark_setup_failure(reason)
    raise
print('detached source commit:', actual, flush=True)


In [ ]:
setup_environment = json.loads(setup_environment_path.read_text(encoding='utf-8'))
if setup_environment.get('status') != 'VALID':
    reason = 'ENVIRONMENT_CHECK:' + setup_environment.get('reason', 'invalid receipt')
    mark_setup_failure(reason)
    raise RuntimeError(reason)
retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
retained['status'] = 'SETUP_COMPLETE_AWAITING_RESULT'
retained['reason'] = None
for case in retained['cases'].values():
    case['status'] = 'PENDING_RUNNER_RESULT'
    for slot in case['slots'].values():
        slot['status'] = 'PENDING_RUNNER_RESULT'
        slot['reason'] = None
write_json(SETUP_SLOTS_PATH, retained)
write_json(SETUP_RECEIPT_PATH, {
    'status': 'SETUP_COMPLETE',
    'source_sha': SOURCE_SHA,
    'output_dir': str(OUTPUT),
    'repo': str(REPO),
    'source_receipt': str(OUTPUT / 'source_receipt.json'),
    'environment_receipt': str(setup_environment_path),
    'python': sys.version,
    'executable': sys.executable,
})
print('setup complete; fixed runner will now create result.json before workers', flush=True)


In [ ]:
RUN_LOG = OUTPUT / 'run.log'
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
command = [
    sys.executable, '-u', '-m',
    'experiments.wan_state_clock.rgb_dct_terminal_gradient_run',
    '--output', str(OUTPUT),
]
completed = logged_run(command, cwd=REPO, env=run_env, check=False, log_path=RUN_LOG)
RESULT_PATH = OUTPUT / 'result.json'
write_json(OUTPUT / 'execution_receipt.json', {
    'command': command,
    'cwd': str(REPO),
    'returncode': completed.returncode,
    'result_path': str(RESULT_PATH),
    'result_exists': RESULT_PATH.is_file(),
})
retained = json.loads(SETUP_SLOTS_PATH.read_text(encoding='utf-8'))
if not RESULT_PATH.is_file():
    retained['status'] = 'RUNNER_FAILED_NO_RESULT'
    retained['reason'] = 'RUNNER_DID_NOT_PRODUCE_RESULT'
    for case in retained['cases'].values():
        case['status'] = 'NOT_RUN_RUNNER_FAILURE'
        for slot in case['slots'].values():
            slot['status'] = 'NOT_RUN_RUNNER_FAILURE'
            slot['reason'] = 'RUNNER_DID_NOT_PRODUCE_RESULT'
    write_json(SETUP_SLOTS_PATH, retained)
    raise FileNotFoundError('runner produced no retained result.json; inspect setup_slots.json and run.log')
retained['status'] = 'SUPERSEDED_BY_RESULT_JSON'
retained['result_path'] = str(RESULT_PATH)
write_json(SETUP_SLOTS_PATH, retained)
print('runner return code:', completed.returncode, flush=True)
print('retained result:', RESULT_PATH, flush=True)


In [ ]:
result = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
if result.get('source_sha') != SOURCE_SHA:
    raise RuntimeError('result source SHA mismatch')
if result.get('fixed_denominator') != FIXED_DENOMINATOR:
    raise RuntimeError('fixed denominator mismatch')
if tuple(result.get('cases', {}).keys()) != CASE_IDS:
    raise RuntimeError('fixed source roster mismatch')
for case_id in CASE_IDS:
    if tuple(result['cases'][case_id].get('slots', {}).keys()) != ARMS:
        raise RuntimeError('fixed arm roster mismatch for ' + case_id)
print('status:', result.get('status'), flush=True)
print('fixed denominator:', result['fixed_denominator'], flush=True)
print(
    'attempted/scored/invalid/pending:',
    result.get('attempted_media_slots'), result.get('scored_media_slots'),
    result.get('invalid_media_slots'), result.get('pending_media_slots'),
    flush=True,
)
for case_id in CASE_IDS:
    case = result['cases'][case_id]
    print('case:', case_id, 'status:', case.get('status'), 'stage:', case.get('stage'), flush=True)
    for arm in ARMS:
        row = case['slots'][arm]
        print(
            case_id, arm, row.get('status'), row.get('positive_groups'),
            row.get('decision'), row.get('reason'), flush=True,
        )
        for layer_name in ('float_rgb', 'rgb8_quantized', 'mp4_rgb24'):
            layer = row.get('layers', {}).get(layer_name, {})
            print(
                '  ', layer_name, layer.get('status'),
                layer.get('positive_groups'), layer.get('decision'),
                layer.get('reason'), flush=True,
            )
print('evidence ceiling:', result.get('evidence_ceiling'), flush=True)
print('read-only result path:', RESULT_PATH, flush=True)
print(json.dumps(result, indent=2, ensure_ascii=False), flush=True)
